# 🤖 1. ReAct 에이전트 기초: File RAG 도구, 시스템 프롬프트 간접 통제, 메모리와 서빙 구조

다중 데이터베이스(Vector · Graph · Text2SQL)를 연동하는 Agentic RAG(`2_Agentic_RAG.ipynb`)로 나아가기 전에,  
이 노트북에서는 **인덱싱 없이 문서 폴더를 직접 탐색하는 File RAG 도구 3종(`list_docs` → `search_docs` → `read_doc`)** 을 바탕으로 **ReAct 에이전트의 3대 기본 구성 요소(도구 · 시스템 프롬프트 · 메모리)** 와 **프로덕션 4계층 서빙 구조**를 처음부터 조립하고 실험합니다.

---

### 🎯 학습 목표
1. 인덱싱 없는 **File RAG 3대 표준 도구(`list_docs` · `search_docs` · `read_doc`)** 와 **샌드박스 경로 검증(`is_relative_to`)** 을 설계하고, 파이썬 함수 코드가 아닌 **이름(`name`) · 설명(`description`) · 인자 스키마(`parameters`)만 LLM 컨텍스트에 전달**되는 원리를 확인합니다.
2. **시스템 프롬프트(System Prompt)** 를 에이전트 자율성에 대한 **간접 통제(Indirect Control)** 수단으로 이해하고, 프롬프트 유무 및 페르소나·작업 규칙 교체에 따라 에이전트의 행동(선제적 도구 호출, 인사 유무, 자기소개, 역질문)이 어떻게 달라지는지 실험합니다.
3. **체크포인터(Checkpointer)** 와 `thread_id`를 연결하여 멀티턴 대화 맥락을 유지하고 세션 간 기억을 격리합니다.
4. `브라우저(User) ⟺ Chat UI ⟺ LLM App(Agent) ⟺ Model Server`로 이어지는 **4계층 서빙 아키텍처**를 해부하고, `Mission 01`을 통해 `app`의 베이스 에이전트인 **`file_rag_agent`** 를 구축합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | 프로젝트 루트 탐색, `.env` 로드, `nest_asyncio`, LLM 초기화 |
| **Part 1** | File RAG 도구 설계와 컨텍스트 전달 원리 | `list_docs` · `search_docs` · `read_doc` + 경로 샌드박스 구현, JSON Schema 해부, 첫 ReAct 루프 |
| **Part 2** | 시스템 프롬프트와 자율성의 간접 통제 | `[실험 2.1]` 프롬프트 유무에 따른 첫 인사 행동 차이 · `[실험 2.2]` HR 매니저 vs 내부감사역 페르소나·규칙 A/B 비교 |
| **Part 3** | 메모리(Checkpointer) 연결 | Stateless 에이전트의 한계 재현, `InMemorySaver` + `thread_id` 멀티턴 대화 및 세션 격리 |
| **Part 4** | 서빙 구조의 해부 | `Browser ⟺ Chat UI ⟺ LLM App ⟺ Model Server` 4계층 아키텍처 및 `app/agents/` 동적 등록 원리 |
| **Part 5** | 정리 및 Mission 01 | File RAG의 유효 범위 vs 한계 브릿지 및 `file_rag_agent` 신설 미션 안내 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 및 LLM 초기화
from app.utils import init_chat_model, normalize_content

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)
print(f"✅ LLM 초기화 완료: {llm.model}")

---
## 🔧 Part 1. File RAG 도구(Tool) 설계와 LLM 컨텍스트 전달 원리

**ReAct (Reasoning + Acting)** 에이전트는 질문을 받자마자 답을 지어내는 대신,  
**생각(Thought) ➔ 도구 호출(Action) ➔ 결과 관찰(Observation)** 루프를 스스로 반복하며 근거를 수집한 뒤 최종 답변을 작성합니다.

이때 별도의 Vector DB 인덱싱 없이도 소규모 마크다운·텍스트 문서 저장소에 즉시 적용할 수 있는 기업 실무 패턴이 **File-based RAG (Agentic File Search)** 입니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    U["사용자 질문"] --> M["LLM (Reasoning)<br/>도구 스키마 기반 판단"]
    M -->|"1. list_docs / 2. search_docs / 3. read_doc"| T["ToolNode (Acting)<br/>샌드박스 내 파일 스캔·열람"]
    T -->|"ToolMessage (Observation)"| M
    M -->|"충분한 근거 확보 시"| A["최종 답변"]

    style M fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style T fill:#fff3e0,stroke:#f57c00,stroke-width:1.5px
```

### 1.1 탐색 대상 문서 저장소(`data/policies/*.md`) 살펴보기

먼저 우리가 도구로 탐색할 `data/policies/` 폴더 안의 공식 사내 규정 마크다운 문서들의 규모(줄 수·용량)를 확인해 봅시다.

In [ ]:
from pathlib import Path

docs_root = Path("data/policies").resolve()
policy_files = sorted(docs_root.glob("*.md"))

print(f"📂 사내 문서 저장소 루트: {docs_root} (총 {len(policy_files)}개 파일)\n")
for fpath in policy_files:
    lines = fpath.read_text(encoding="utf-8").splitlines()
    size_kb = fpath.stat().st_size / 1024
    print(f"  - {fpath.name:<34} | {len(lines):>4}줄 ({size_kb:>4.1f}KB) | {lines[0].strip()}")

print("\n" + "=" * 80)
sample_lines = (docs_root / "2026_국내여비교통비지급규정.md").read_text(encoding="utf-8").splitlines()[:20]
print("📄 [미리보기] 2026_국내여비교통비지급규정.md (1~20줄)\n")
print("\n".join(f"{i:4d} | {ln}" for i, ln in enumerate(sample_lines, 1)))

### 1.2 엔터프라이즈 File RAG 3대 표준 도구 디자인하기

특정 문서 양식(`### 제N조` 등)에 종속되지 않고 어떤 마크다운·텍스트 문서 폴더에나 재사용할 수 있도록 현업 에이전트(Claude Code, Codex 등)와 동일한 **3단계 탐색 프리미티브**와 **샌드박스 보안 경계**를 설계합니다.

| 도구명 | 역할 | 기업 실무 설계 포인트 |
|:---|:---|:---|
| **`list_docs`** | 문서 목록·크기·첫 줄 제목 조회 | 저장소에 어떤 문서들이 비치되어 있는지 전체 지형 파악 |
| **`search_docs`** | 다중 파일 키워드/정규식 Grep 검색 | **파일명을 미리 몰라도** 전체 문서를 가로질러 `파일명 : 줄번호 : 섹션 : 스니펫` 특정 |
| **`read_doc`** | 지정 줄 구간(`start_line ~ end_line`) 열람 | 스니펫 주변의 표·단서 조항을 잘림 없이 열람 + **`Path.resolve().is_relative_to(DOCS_ROOT)`로 상위 경로(`../../.env`) 접근 원천 차단** |

`@tool(parse_docstring=True)` 데코레이터를 사용하면 파이썬 함수의 **Type Hint**와 **Google 스타일 Docstring(`Args:`)** 을 자동으로 파싱해 표준 도구 스키마로 변환합니다.

In [ ]:
import re
from pathlib import Path
from typing import Optional
from langchain_core.tools import tool

DOCS_ROOT = Path("data/policies").resolve()
ALLOWED_EXTS = {".md", ".txt"}
MAX_READ_LINES = 150


def _resolve_safe_path(rel_path: str) -> tuple[Optional[Path], Optional[str]]:
    """문서 루트(DOCS_ROOT) 외부로의 경로 탈출(Path Traversal)을 차단합니다."""
    raw = (rel_path or "").strip().strip("'\"")
    candidate = (DOCS_ROOT / raw).resolve()
    if not candidate.is_relative_to(DOCS_ROOT):
        return None, f"[보안 차단] 허용된 문서 루트(data/policies) 외부 경로에는 접근할 수 없습니다: '{rel_path}'"
    return candidate, None


def _iter_doc_files(base_dir: Path) -> list[Path]:
    if not base_dir.exists():
        return []
    return sorted(
        p for p in base_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in ALLOWED_EXTS and p.resolve().is_relative_to(DOCS_ROOT)
    )


@tool(parse_docstring=True)
def list_docs(subdir: str = "") -> str:
    """허용된 사내 문서 저장소(data/policies) 내 문서 파일 목록, 줄 수, 크기, 첫 줄 제목을 조회합니다.

    Args:
        subdir: 조회할 하위 폴더 상대 경로. 생략하면 저장소 전체 문서 목록을 반환합니다.
    """
    target_dir, err = _resolve_safe_path(subdir)
    if err:
        return err
    if not target_dir.exists() or not target_dir.is_dir():
        return f"[오류] 디렉터리를 찾을 수 없습니다: '{subdir}'"

    files = _iter_doc_files(target_dir)
    if not files:
        return "[알림] 조회 가능한 문서 파일(.md, .txt)이 없습니다."

    rows = []
    for fpath in files:
        rel = fpath.relative_to(DOCS_ROOT).as_posix()
        lines = fpath.read_text(encoding="utf-8", errors="replace").splitlines()
        first_title = next((ln.strip().lstrip("# ").strip() for ln in lines if ln.strip()), "(제목 없음)")
        size_kb = fpath.stat().st_size / 1024
        rows.append(f"- {rel} ({len(lines)}줄, {size_kb:.1f}KB) : {first_title}")

    return f"📚 문서 저장소 목록 (총 {len(rows)}개 파일):\n" + "\n".join(rows)


@tool(parse_docstring=True)
def search_docs(query: str, path_filter: Optional[str] = None) -> str:
    """문서 저장소 전체(또는 특정 파일)를 가로질러 키워드나 정규식이 등장하는 파일명·줄번호·섹션·스니펫을 검색합니다.

    어느 파일의 몇 번째 줄에 관련 내용이 있는지 모를 때 read_doc 호출 전에 먼저 사용하세요.

    Args:
        query: 검색할 핵심 키워드(공백으로 여러 단어 입력 가능) 또는 정규표현식 (예: '숙박비 부산', '제14조', 'GPU.*회수')
        path_filter: 특정 파일명이나 하위 경로 일부로 범위를 좁힐 때 지정 (예: '여비', '보안'). 생략 시 전체 문서 검색
    """
    q = (query or "").strip()
    if not q:
        return "[오류] 검색어(query)를 비워둘 수 없습니다."

    if path_filter and ".." in path_filter:
        _, err = _resolve_safe_path(path_filter)
        if err:
            return err

    files = _iter_doc_files(DOCS_ROOT)
    if path_filter:
        pf = path_filter.strip().lower()
        files = [f for f in files if pf in f.relative_to(DOCS_ROOT).as_posix().lower()]
    if not files:
        return f"[알림] path_filter='{path_filter}' 조건에 해당하는 문서가 없습니다."

    terms = [t for t in q.split() if t]
    regex = None
    if len(terms) == 1 and any(ch in q for ch in ".*+?[]()|^$"):
        try:
            regex = re.compile(q, re.IGNORECASE)
        except re.error:
            regex = None

    hits = []
    for fpath in files:
        rel = fpath.relative_to(DOCS_ROOT).as_posix()
        lines = fpath.read_text(encoding="utf-8", errors="replace").splitlines()
        current_section = "(문서 상단)"
        last_hit_line = -10

        for idx, line in enumerate(lines):
            stripped = line.strip()
            if stripped.startswith("#"):
                current_section = stripped.lstrip("#").strip()

            window_text = " ".join(lines[max(0, idx - 1): min(len(lines), idx + 2)]).lower()
            if regex:
                matched_count = 1 if regex.search(line) else 0
            else:
                line_lower = line.lower()
                if not any(t.lower() in line_lower for t in terms):
                    continue
                matched_count = sum(1 for t in terms if t.lower() in window_text)

            if matched_count > 0 and (idx - last_hit_line) >= 3:
                last_hit_line = idx
                start_s = max(0, idx - 1)
                end_s = min(len(lines), idx + 2)
                snippet = " / ".join(ln.strip() for ln in lines[start_s:end_s] if ln.strip())
                hits.append((matched_count, rel, idx + 1, current_section, snippet[:180]))

    if not hits:
        return f"[검색 결과 없음] '{query}'와 일치하는 구절을 찾지 못했습니다. 다른 동의어나 상위 키워드로 다시 검색하거나 list_docs로 문서 목록을 확인하세요."

    hits.sort(key=lambda x: (-x[0], x[1], x[2]))
    out = [f"🔎 '{query}' 검색 결과 (총 {len(hits)}건 중 상위 {min(len(hits), 10)}건):"]
    for score, rel, line_no, sec, snippet in hits[:10]:
        out.append(f"- 📄 {rel} (L{line_no}, 섹션: {sec})\n  └ {snippet}")
    out.append("\n💡 상세 표·단서 조항 확인이 필요하면 read_doc(filename, start_line, end_line)으로 해당 줄 주변을 펼쳐 읽으세요.")
    return "\n".join(out)


@tool(parse_docstring=True)
def read_doc(filename: str, start_line: int = 1, end_line: int = 100) -> str:
    """지정한 문서 파일의 특정 줄 구간(start_line ~ end_line) 원문을 줄 번호와 함께 정밀 열람합니다.

    Args:
        filename: 열람할 문서 상대 경로 또는 파일명 (예: '2026_국내여비교통비지급규정.md')
        start_line: 읽기 시작할 줄 번호 (1부터 시작, 기본값 1)
        end_line: 읽기 종료할 줄 번호 (1회 최대 150줄까지 열람 가능, 기본값 100)
    """
    fpath, err = _resolve_safe_path(filename)
    if err:
        return err

    if not fpath.is_file():
        stem = Path(filename.strip()).stem.lower()
        candidates = [f for f in _iter_doc_files(DOCS_ROOT) if stem and stem in f.name.lower()]
        if len(candidates) == 1:
            fpath = candidates[0]
        else:
            return f"[오류] '{filename}' 파일을 찾을 수 없습니다. list_docs()로 정확한 파일명을 먼저 확인하세요."

    lines = fpath.read_text(encoding="utf-8", errors="replace").splitlines()
    total = len(lines)
    s = max(1, int(start_line))
    e = min(total, max(s, int(end_line)), s + MAX_READ_LINES - 1)

    if s > total:
        return f"[알림] '{fpath.name}' 파일은 전체 {total}줄입니다. start_line({s})을 {total} 이하로 지정하세요."

    rel = fpath.relative_to(DOCS_ROOT).as_posix()
    body = "\n".join(f"{ln_no:4d} | {lines[ln_no - 1]}" for ln_no in range(s, e + 1))
    return f"[파일: {rel} | 열람 구간: {s}~{e}줄 / 전체 {total}줄]\n{body}"


file_rag_tools = [list_docs, search_docs, read_doc]

# 1) 다중 파일 가로지르기 검색(search_docs) -> 2) 라인 구간 열람(read_doc) -> 3) 샌드박스 보안 차단 테스트
print(search_docs.invoke({"query": "유휴 GPU 회수"}))
print("\n" + "=" * 80)
print(read_doc.invoke({"filename": "2026_클라우드및GPU인프라운영지침.md", "start_line": 78, "end_line": 102}))
print("\n" + "=" * 80)
print("🔒 [샌드박스 경계 테스트] 상위 경로(.env) 탈출 시도 결과:")
print(read_doc.invoke({"filename": "../../.env", "start_line": 1, "end_line": 10}))

### 1.3 LLM 컨텍스트에는 무엇이 전달될까? (함수 코드 격리 원리)

에이전트에 도구를 연결할 때 반드시 이해해야 할 핵심 원리가 있습니다.

> 💡 **핵심 원리: LLM은 파이썬 함수 본문 코드를 단 한 줄도 보지 못합니다!**  
> `create_agent(model=llm, tools=file_rag_tools)`를 실행하면, LangChain은 함수 내부 로직(`rglob`, `is_relative_to`, 정규식 매칭 등)을 제외한 채 오직 아래 **3가지 메타데이터만 JSON Schema로 직렬화**하여 LLM 컨텍스트에 주입합니다.
> 1. **도구 이름 (`name`)**: 함수 이름 (`list_docs`, `search_docs`, `read_doc`)
> 2. **도구 설명 (`description`)**: Docstring 상단 설명 — *"언제, 어떤 순서로 이 도구를 써야 하는가"*
> 3. **인자 정보 (`parameters`)**: Type Hint + `Args:` 설명으로 생성된 JSON Schema — *"어떤 인자를 어떤 형식으로 넘겨야 하는가"*

실제로 Model Server로 전송되는 JSON 구조를 직접 눈으로 확인해 봅시다.

In [ ]:
import json
import inspect
from langchain_core.utils.function_calling import convert_to_openai_tool

raw_source = inspect.getsource(search_docs.func)

print(f"🐍 [애플리케이션 런타임 전용] search_docs 파이썬 코드 길이: {len(raw_source.splitlines())}줄 (LLM에는 절대 전달되지 않음!)")
print("-" * 80)
print("📨 [LLM 컨텍스트 전달본] Model Server로 전송되는 실제 Tool JSON Schema:\n")
for t in file_rag_tools:
    print(json.dumps(convert_to_openai_tool(t), ensure_ascii=False, indent=2))
    print("-" * 80)

위 출력에서 볼 수 있듯, 도구 내부 구현이 50줄이든 5,000줄이든 LLM의 컨텍스트 토큰은 **오직 위 JSON 스키마 크기만큼만 소모**됩니다.  
반대로 말하면, **함수 이름·설명·인자 설명이 모호하면 아무리 파이썬 로직을 정교하게 짜도 LLM은 언제 어떤 인자로 도구를 호출해야 할지 알 수 없습니다.**

### 1.4 `create_agent`로 첫 File RAG 에이전트 조립 및 실행 궤적 확인

이제 시스템 프롬프트나 메모리 없이 **`model`과 `file_rag_tools`만 연결한 순수 ReAct 에이전트**에게 **어느 규정 파일에 있는지 파일명을 알려주지 않은 채** 질문을 던지고, 에이전트가 스스로 `search_docs` ➔ `read_doc` 순서로 원문을 찾아 답하는 궤적을 확인해 봅시다.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

basic_agent = create_agent(
    model=llm,
    tools=file_rag_tools,
)


def run_and_trace(agent, question: str, config: dict | None = None, show_tool_output: bool = True):
    """에이전트를 실행하고 이번 턴의 도구 호출 궤적(Trajectory)과 최종 답변을 출력합니다."""
    res = agent.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    messages = res["messages"]
    last_human_idx = max(i for i, m in enumerate(messages) if m.type == "human")
    turn_msgs = messages[last_human_idx:]

    print(f"❓ 사용자 입력: {question}")
    print("-" * 80)

    tool_call_count = 0
    for m in turn_msgs[1:]:
        if m.type == "ai" and getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                tool_call_count += 1
                print(f"  🔄 [Action {tool_call_count}] {tc['name']}({tc['args']})")
        elif m.type == "tool" and show_tool_output:
            snippet = normalize_content(m.content).replace("\n", " ")[:140]
            print(f"  📥 [Observation] ({m.name}) {snippet}...")

    print(f"  📊 총 도구 호출 횟수: {tool_call_count}회")
    print("-" * 80)
    print("💬 에이전트 답변:\n")
    print(normalize_content(turn_msgs[-1].content))
    return res


_ = run_and_trace(
    basic_agent,
    "국내 출장 시 직급별 1박 숙박비 상한액은 얼마이고, 출장비 정산은 복귀 후 며칠 이내에 마쳐야 해?",
)

---
## 🎯 Part 2. 시스템 프롬프트(System Prompt): 에이전트 자율성에 대한 간접 통제

전통적인 소프트웨어는 `if/else` 분기문으로 실행 순서를 **직접 통제(Direct Control)** 합니다.  
반면 ReAct 에이전트는 매 순간 *"도구를 부를까 말까? 어떤 도구를 먼저 부를까? 바로 답할까, 조건을 되물을까?"* 를 LLM이 자율적으로 결정합니다.

**시스템 프롬프트(`system_prompt`)** 는 파이썬 코드를 한 줄도 수정하지 않고 에이전트의 **자율성(Autonomy)에 경계와 방향을 부여하는 간접 통제(Indirect Control) 장치**입니다. 시스템 프롬프트에는 크게 두 가지가 기술됩니다.

1. **핵심 페르소나 (Persona & Identity)**: 에이전트가 누구인지(소속·역할), 어떤 말투와 태도(인사 유무, 친절함 vs 엄격함)로 소통할지 정의합니다.
2. **작업 규칙 (Operating Rules)**: 언제 어떤 도구를 선제적으로 호출할지(`search_docs` 후 반드시 `read_doc`으로 원문 검증 등), 모호한 질문에 어떻게 대처할지, 어떤 출력 양식과 출처 표기 원칙을 지킬지 규정합니다.

아래 2단계 실험을 통해 **완전히 동일한 모델(`llm`)과 동일한 도구(`file_rag_tools`)** 상태에서 시스템 프롬프트만으로 에이전트의 행동이 어떻게 달라지는지 직접 비교해 보겠습니다.

---
### 2.1 `[실험 1]` 시스템 프롬프트 유무 비교: 첫 인사와 자기소개 행동

사용자가 구체적인 질문 대신 **`"안녕하세요! 반가워요."`** 라고 첫 인사를 건넸을 때,  
시스템 프롬프트가 없는 `basic_agent`와 사내 문서 안내 데스크 규칙이 부여된 `desk_agent`의 행동을 비교해 봅시다.

In [ ]:
greeting_input = "안녕하세요! 반가워요."

print("🧪 [Case A] 시스템 프롬프트가 없는 기본 에이전트 (basic_agent)")
print("=" * 80)
_ = run_and_trace(basic_agent, greeting_input, show_tool_output=False)

시스템 프롬프트가 없는 `basic_agent`는 도구를 **0회 호출**하고 일반적인 AI 어시스턴트로서 짧은 인사만 건넵니다. 자신이 `(주)넥스트AI` 소속인지, 저장소에 어떤 사내 문서들이 비치되어 있는지 전혀 알지 못합니다.

이번에는 **페르소나(사내 문서 안내 데스크)** 와 **첫 인사 시 `list_docs` 선제 조회 및 File RAG 3단계 탐색 규칙**을 담은 시스템 프롬프트를 주입해 보겠습니다.

In [ ]:
FILE_RAG_DESK_PROMPT = """당신은 (주)넥스트AI의 공식 '사내 문서 안내 데스크(File RAG)' 에이전트입니다.
임직원의 사내 규정·지침 문의에 대해 파일 탐색 도구(list_docs, search_docs, read_doc)만을 근거로 정확하고 명확하게 한국어로 답변하세요.

[핵심 작업 규칙]
1. 사용자가 첫 인사를 건네거나 어떤 도움을 줄 수 있는지 물으면, 추측하지 말고 반드시 먼저 `list_docs` 도구를 호출하여 현재 비치된 문서 목록을 확인한 뒤 자기소개와 함께 안내 가능한 문서를 브리핑하세요.
2. 기준·수치·절차 질문을 받으면 `search_docs`로 위치를 찾은 뒤 반드시 `read_doc`으로 해당 줄 주변 원문(표·단서 조항)을 확인하고 문서명과 조항 번호(예: 제14조)를 함께 안내하세요.
3. 사내 문서에서 확인되지 않는 내용은 절대 지어내지 말고 "현재 비치된 사내 문서에서 확인되지 않는 사항입니다."라고 안내하세요.
"""

desk_agent = create_agent(
    model=llm,
    tools=file_rag_tools,
    system_prompt=FILE_RAG_DESK_PROMPT,
)

print("🧪 [Case B] 사내 문서 안내 데스크 프롬프트 적용 에이전트 (desk_agent)")
print("=" * 80)
_ = run_and_trace(desk_agent, greeting_input)

똑같은 `"안녕하세요! 반가워요."` 입력에 대해, `desk_agent`는 스스로 **`list_docs()`를 먼저 호출**한 뒤 `(주)넥스트AI 사내 문서 안내 데스크`로서 인사하고 실제 비치된 **5대 공식 규정·지침 문서**를 정확히 브리핑합니다!

---
### 2.2 `[실험 2]` 페르소나 & 작업 규칙 A/B 스왑: `친절한 HR 온보딩 매니저` vs `원칙주의 내부감사역`

이번에는 도메인 현업에서 자주 만나는 두 가지 상반된 역할로 시스템 프롬프트를 교체해 보겠습니다.

| 비교 항목 | 😊 페르소나 A: `친절한 HR 온보딩 매니저` | ⚖️ 페르소나 B: `원칙주의 내부감사역` |
|:---|:---|:---|
| **인사 및 자기소개** | 항상 반가운 인사와 자기소개(`HR 온보딩 매니저 한결`)로 시작 | **인사말·자기소개·감정 표현 전면 금지** (첫 줄부터 본론) |
| **답변 양식** | 이해하기 쉬운 풀어서 설명 + 실무 팁 안내 | **`[판정 결론]` / `[근거 규정]` / `[세부 기준]`** 3단 감사 보고서 고정 |
| **조건 누락 질문 대응**<br/>*(예: 직급·지역 없는 숙박비 질문)* | 당황하지 않도록 **전 직급·권역별 한도표를 한눈에 요약** 제공 | 임의 단정 및 전체 나열 금지 ➔ **대상자의 직급과 출장 권역을 먼저 역질문(Clarification)** |

In [ ]:
HR_MANAGER_PROMPT = """당신은 (주)넥스트AI 신규 입사자의 빠른 적응을 돕는 친절한 'HR 온보딩 매니저 한결'입니다.

[페르소나 및 작업 규칙]
1. 모든 답변의 첫 문장은 반드시 반가운 인사와 짧은 자기소개(예: "안녕하세요! (주)넥스트AI HR 온보딩 매니저 한결입니다 😊")로 시작하세요.
2. 규정 질문에는 반드시 `search_docs`와 `read_doc` 도구로 실제 조항 원문을 확인해 쉽게 풀어서 설명하고 마지막에 실무 정산 팁을 한 줄 덧붙이세요.
3. 만약 질문에 직급이나 출장 지역 같은 세부 조건이 빠져 있더라도, 질문자가 한 번에 파악할 수 있도록 전 직급·지역별 기준을 친절하게 정리해 안내하세요.
"""

STRICT_AUDITOR_PROMPT = """당신은 (주)넥스트AI 감사실 소속 '원칙주의 내부감사역'입니다.

[페르소나 및 작업 규칙]
1. 인사말, 자기소개, 감사 표현, 이모지를 절대 사용하지 마십시오. 첫 글자부터 즉시 감사 결과 또는 확인 요청 사항만 기술하십시오.
2. 규정 수치를 답할 때는 반드시 `search_docs`와 `read_doc` 도구로 원문을 검증한 뒤 아래 3단 양식으로만 출력하십시오:
   [판정 결론] 핵심 금액 및 가부 한 줄 요약
   [근거 규정] 문서명, 문서번호 및 조항 번호
   [세부 기준] 원문 수치 및 산정 조건
3. 직급(임원/팀장/팀원)이나 출장 권역(서울·수도권/광역시/기타 지방)에 따라 한도가 달라지는 항목에서 해당 조건이 질문에 명시되지 않은 경우, 전체 표를 나열하거나 임의 추정하지 말고 조항 확인 후 `[판정 보류 — 추가 정보 필요]` 제목 아래 대상자의 직급과 출장 권역을 먼저 되물어 특정하십시오.
"""

hr_manager_agent = create_agent(model=llm, tools=file_rag_tools, system_prompt=HR_MANAGER_PROMPT)
strict_auditor_agent = create_agent(model=llm, tools=file_rag_tools, system_prompt=STRICT_AUDITOR_PROMPT)

print("✅ 두 페르소나 에이전트 준비 완료 (동일 모델 · 동일 도구 · 다른 시스템 프롬프트)")

먼저 **조건이 명확한 질문**(`"팀원이 당일 출장을 가면 일비와 식비는 얼마를 받나요?"`)을 두 에이전트에게 동일하게 던져 인사의 유무와 보고 양식 차이를 확인합니다.

In [ ]:
specific_q = "팀원이 당일 출장을 가면 일비와 식비는 얼마를 받나요?"

print("😊 [페르소나 A: HR 온보딩 매니저] — 조건이 명확한 질문")
print("=" * 80)
_ = run_and_trace(hr_manager_agent, specific_q, show_tool_output=False)

print("\n" + "#" * 80 + "\n")

print("⚖️ [페르소나 B: 원칙주의 내부감사역] — 조건이 명확한 질문")
print("=" * 80)
_ = run_and_trace(strict_auditor_agent, specific_q, show_tool_output=False)

이번에는 **핵심 조건(직급·출장 지역)이 빠진 모호한 질문**(`"국내 출장 갈 때 1박 숙박비 한도가 얼마인가요?"`)을 던져, 시스템 프롬프트가 단순 말투를 넘어 **에이전트의 의사결정 행동(전체 요약 안내 vs 역질문을 통한 조건 특정)** 까지 어떻게 간접 통제하는지 확인합니다.

In [ ]:
ambiguous_q = "국내 출장 갈 때 1박 숙박비 한도가 얼마인가요?"

print("😊 [페르소나 A: HR 온보딩 매니저] — 조건이 누락된 모호한 질문")
print("=" * 80)
_ = run_and_trace(hr_manager_agent, ambiguous_q, show_tool_output=False)

print("\n" + "#" * 80 + "\n")

print("⚖️ [페르소나 B: 원칙주의 내부감사역] — 조건이 누락된 모호한 질문")
print("=" * 80)
_ = run_and_trace(strict_auditor_agent, ambiguous_q, show_tool_output=False)

---
## 🧠 Part 3. 메모리(Memory): Checkpointer를 통한 멀티턴 대화 연결

지금까지 만든 에이전트는 한 번의 `.invoke()` 호출이 끝나면 이전 대화 상태를 모두 잊어버리는 **무상태(Stateless)** 구조입니다.  
실제 채팅 서비스처럼 *"그럼 팀원은 얼마야?"* 같은 후속 질문을 이해하려면 **체크포인터(Checkpointer)** 를 연결해야 합니다.

### 3.1 체크포인터가 없을 때 발생하는 문제 확인

In [ ]:
print("=== ❌ [Stateless 에이전트] Turn 1 ===")
_ = run_and_trace(desk_agent, "서울로 국내 출장 갈 때 팀장의 1박 숙박비 한도는 얼마야?", show_tool_output=False)

print("\n=== ❌ [Stateless 에이전트] Turn 2 (이전 맥락 유실) ===")
_ = run_and_trace(desk_agent, "그럼 거기 갈 때 팀원은 얼마야? 방금 말한 금액과 차액도 계산해줘.", show_tool_output=False)

Turn 2에서는 `"서울 국내 출장 1박 숙박비"`라는 전제가 사라졌기 때문에 에이전트가 어떤 항목의 팀원 기준을 묻는지 알지 못하거나 엉뚱하게 재질문하게 됩니다.

### 3.2 `InMemorySaver`와 `thread_id`로 멀티턴 기억 연결하기

LangGraph 런타임의 **Checkpointer**는 매 턴이 끝날 때마다 `thread_id`(대화방 ID)를 키로 삼아 전체 메시지 상태(`messages`)를 저장하고, 다음 턴 호출 시 자동으로 이전 대화를 복원해 이어 붙입니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 800, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    T1["Turn 1 입력<br/>(thread_id='kim')"] --> A1["ReAct 실행"] --> CP[("Checkpointer<br/>State 저장소")]
    CP -->|"이전 messages 자동 복원"| A2["ReAct 실행"]
    T2["Turn 2: '그럼 거기 갈 때 팀원은?'<br/>(thread_id='kim')"] --> A2 --> OUT["맥락 유지 답변<br/>(13만원 - 10만원 = 3만원)"]

    style CP fill:#fff3e0,stroke:#f57c00,stroke-width:1.5px
    style OUT fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
```

- **노트북 실험용**: `InMemorySaver` (파이썬 프로세스 메모리에 저장)
- **프로덕션 서버용 (`app/server.py`, `file_rag_agent.py`)**: `AsyncSqliteSaver` (`app/database/checkpoints.db` 파일에 영구 저장 — 서버 재시작 후에도 대화 유지)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_saver = InMemorySaver()

memory_agent = create_agent(
    model=llm,
    tools=file_rag_tools,
    system_prompt=FILE_RAG_DESK_PROMPT,
    checkpointer=memory_saver,
)

kim_session = {"configurable": {"thread_id": "session_kim"}}
park_session = {"configurable": {"thread_id": "session_park"}}

print("=== 🟢 [session_kim | Turn 1] ===")
_ = run_and_trace(
    memory_agent,
    "서울로 국내 출장 갈 때 팀장의 1박 숙박비 한도는 얼마야?",
    config=kim_session,
    show_tool_output=False,
)

print("\n=== 🟢 [session_kim | Turn 2 — 생략된 맥락 복원 & 차액 계산] ===")
_ = run_and_trace(
    memory_agent,
    "그럼 거기 갈 때 팀원은 얼마야? 방금 말한 금액과 차액도 계산해줘.",
    config=kim_session,
    show_tool_output=False,
)

print("\n=== 🔒 [session_park | 다른 thread_id — 세션 격리 확인] ===")
_ = run_and_trace(
    memory_agent,
    "우리가 방금 어느 지역 출장 숙박비를 이야기하고 있었지?",
    config=park_session,
    show_tool_output=False,
)

---
## 🏢 Part 4. 서빙 구조의 해부: 4계층 에이전트 아키텍처

주피터 노트북에서 만든 `create_agent(model, tools, system_prompt, checkpointer)` 코드는 실제 웹 서비스에서 어떻게 동작할까요?  
프로덕션 LLM 애플리케이션은 아래 **4개의 독립된 계층(`브라우저 ⟺ Chat UI ⟺ LLM App ⟺ Model Server`)** 이 양방향으로 통신하며 구동됩니다.

```mermaid
%%{init: {'flowchart': {'wrappingWidth': 850, 'padding': 20}, 'themeCSS': 'foreignObject { overflow: visible !important; }'}}%%
flowchart LR
    B["🌐 브라우저 (User)<br/>localhost:8080<br/>로그인 · 프로필 선택"]
    UI["💬 Chat UI<br/>Chainlit (:8080)<br/>app/chainlit_ui.py"]
    APP["🧠 LLM App (Agent Server)<br/>FastAPI + LangGraph (:8000)<br/>app/server.py · app/agents/*"]
    MS["⚡ Model Server<br/>Gemini / OpenAI / vLLM<br/>무상태(Stateless) 추론 엔진"]

    B <-->|"HTTP / WebSocket<br/>채팅 입출력 · Step 표시"| UI
    UI <-->|"GET /agents<br/>POST /agents/{name}/stream (SSE)"| APP
    APP <-->|"HTTPS JSON API<br/>Messages + Tool Schema ⇄ tool_calls"| MS

    APP --- DB[("📂 data/policies/*.md<br/>💾 checkpoints.db")]

    style B fill:#e3f2fd,stroke:#1e88e5,stroke-width:1.5px
    style UI fill:#e8f5e9,stroke:#2e7d32,stroke-width:1.5px
    style APP fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style MS fill:#f3e5f5,stroke:#8e24aa,stroke-width:1.5px
```

> 💡 **아키텍처 핵심 포인트**  
> **Model Server**는 시스템 프롬프트·대화 이력·도구 스키마(JSON)를 받아 다음 행동(`tool_calls`)이나 답변 텍스트만 추론할 뿐 사내 파일이나 DB에 직접 접근하지 않습니다. **실제 사내 파일 스캔·열람(`file_rag_tools`), 경로 샌드박스 검증, 체크포인터 저장(`checkpoints.db`)은 모두 LLM App(Agent Server) 내부에서 실행**됩니다.

| 계층 | 구성 파일 / 엔드포인트 | 통신 프로토콜 | 핵심 역할 |
|:---|:---|:---|:---|
| **1. 브라우저 (User)** | `http://localhost:8080` | HTTP / WebSocket | 사용자 로그인(`user / 1234`), 에이전트 프로필 선택, 스트리밍 답변 및 도구 실행 Step 확인 |
| **2. Chat UI** | `app/chainlit_ui.py` (`:8080`) | REST (`GET /agents`) · SSE | 기동 시 서버 에이전트 목록 자동 조회, 세션별 `thread_id` 관리, SSE 이벤트 채팅 화면 렌더링 |
| **3. LLM App (Agent)** | `app/server.py` · `app/agents/*` (`:8000`) | `POST /agents/{name}/stream` | **LangGraph ReAct 루프 오케스트레이션**, 파이썬 도구 실제 실행(`data/policies/*.md`), `AsyncSqliteSaver` 영구 메모리 관리 |
| **4. Model Server** | Gemini / OpenAI API / 사내 vLLM | HTTPS JSON API | 전달받은 메시지 + 도구 JSON 스키마 기반 무상태(Stateless) 추론 (`tool_calls` JSON 또는 응답 토큰 반환) |

---
### 4.1 `app/server.py`의 에이전트 자동 탐색(Dynamic Discovery) 규약

우리 실습 프로젝트의 백엔드(`app/server.py`)는 새로운 에이전트를 추가할 때마다 서버 라우팅 코드나 UI 코드를 고치지 않아도 되도록 **플러그인 자동 스캔 구조**로 설계되어 있습니다.

`app/agents/` 폴더 아래에 파이썬 파일(예: `app/agents/file_rag_agent.py`)을 만들고 다음 **2가지 심볼**만 정의하면, 서버 기동 시 자동으로 감지되어 Chainlit 웹 UI 상단 프로필 목록에 즉시 나타납니다!

1. **`AGENT_METADATA`** (`dict`): UI 드롭다운에 표시할 에이전트 이름(`name`)과 설명(`description`)
2. **`async def create_agent_executor(checkpointer=None)`**: 모델 + 도구 + 시스템 프롬프트 + 체크포인터를 결합해 `create_agent(..., context_schema=AgentContext)` 인스턴스를 반환하는 비동기 팩토리 함수

현재 `app/agents/` 디렉터리에 어떤 에이전트들이 등록되어 있는지 확인해 봅시다.

In [ ]:
from app.server import api_list_agents

registered = api_list_agents()
print("📋 현재 app/server.py가 감지한 에이전트 목록:\n")
for item in registered:
    print(f"  🤖 {item['name']:<18} : {item['description']}")

---
## 🏁 Part 5. 정리 및 Mission 01 안내

### 📌 ReAct 에이전트 3대 구성 요소 핵심 요약

| 구성 요소 | 코드 인터페이스 | 핵심 역할 |
|:---|:---|:---|
| **1. 도구 (Tool)** | `@tool(parse_docstring=True)` | 에이전트의 손과 발. **함수 코드는 격리**되고 오직 `name` · `description` · `parameters` JSON 스키마만 Model Server로 전달됨 |
| **2. 시스템 프롬프트** | `system_prompt=...` | 에이전트 자율성에 대한 **간접 통제(Indirect Control)**. 페르소나(인사·말투·역할)와 작업 규칙(선제 행동·출력 양식·역질문)을 조향 |
| **3. 메모리 (Checkpointer)** | `InMemorySaver` / `AsyncSqliteSaver` | `thread_id` 단위로 대화 상태(`messages`)를 저장·복원하여 무상태 LLM 위에 멀티턴 문맥을 부여 |

---
### 🔗 다음 단계 예고: File RAG는 어디까지 유효하고, 언제 Vector · Graph · SQL이 필요한가?

오늘 만든 **`file_rag_agent` (`list_docs` → `search_docs` → `read_doc`)** 는 별도의 DB 구축 없이도 수십 개 이내의 마크다운·텍스트 규정집에서는 놀라울 만큼 정확하게 동작합니다. 실제로 문서 양이 적고 수시로 개정되는 팀 위키·운영 가이드에는 File RAG가 가장 경제적인 첫 선택지입니다.

하지만 다음 세 가지 순간부터 File RAG만으로는 해결할 수 없는 벽에 부딪히며, 이에 대한 구체적인 **저장소 선택 기준**과 해법을 `2_Agentic_RAG.ipynb` ~ `4_Text2SQL.ipynb`에서 본격적으로 다룹니다.

1. **어휘 불일치 & 대용량 비정형 문서**: 사용자가 `"과장급 출장비"`라고 물으면 원문에는 `"선임/책임"`만 있어 키워드 Grep(`search_docs`)이 빗나가며, 수백 페이지 영문 PDF 백서(4,600+ 청크)는 실시간 파일 스캔이 불가능합니다 ➔ **Vector + BM25 Hybrid Search (`3_Vector_and_Graph_RAG`)**
2. **복합 엔티티 관계 추론**: 여러 문서에 흩어진 `조직 ↔ 인력 ↔ 프로젝트 ↔ 인프라`의 다단계 의존성과 영향도 분석 ➔ **Knowledge Graph & GraphRAG (`3_Vector_and_Graph_RAG`)**
3. **정확한 수치 집계와 필터링**: 수천~수만 행의 ERP 예산·집행·발주 테이블에 대한 합계(`SUM`)·비율·조건 조회 ➔ **시멘틱 레이어 기반 Text2SQL (`4_Text2SQL`)**

---
### 🎯 Mission 01: `file_rag_agent` 신설 및 Chainlit Chat UI 실전 검증

이제 `missions/01_missions.md` 문서를 열고 첫 번째 실습 미션을 수행하세요!

1. **File RAG 도구 3종 완성 (`app/tools/file_rag_tools.py`)**: 이 노트북 Part 1에서 만든 `list_docs`, `search_docs`, `read_doc` 도구를 구현합니다.
2. **베이스 에이전트 신설 (`app/agents/file_rag_agent.py`)**: `AGENT_METADATA`, `FILE_RAG_SYSTEM_PROMPT`, `create_agent_executor`를 작성해 `app/agents/`에 등록합니다 (`file_rag_agent` ➔ `tool_rag_agent` ➔ `skill_rag_agent` 3단계 진화의 첫 베이스 에이전트).
3. **서버 & Chat UI 대화 검증**: FastAPI 서버(`:8000`)와 Chainlit UI(`:8080`)를 띄우고 **`file_rag_agent`** 를 선택해 **첫 인사 목록 브리핑 ➔ 다중 파일 가로지르기 검색(`search_docs` → `read_doc`) ➔ 멀티턴 대화 ➔ 어휘 불일치 한계 체감**까지 직접 확인합니다!

In [ ]:
with open("missions/01_missions.md", "r", encoding="utf-8") as f:
    mission_preview = "".join([next(f) for _ in range(32)])
print(mission_preview)
print("... (전체 가이드는 missions/01_missions.md 파일에서 확인하세요!)")